# Lot 2 — Moteur sémantique

On compare trois méthodes sur les 600 critiques anglaises de référence. Pour cinq requêtes en français ou en anglais, chaque méthode classe les mêmes vins par similarité cosinus avec leur description. Le classement utilise uniquement le texte des critiques. Les filtres et le score de recommandation relèvent du lot 3.

Après `python -m pip install -r requirements.txt` à la racine du projet, ouvrir ce notebook avec le noyau Python de cet environnement et exécuter toutes les cellules. La première exécution télécharge les ressources NLTK et le modèle SBERT, puis crée le cache des vecteurs des vins.

In [1]:
import sys
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from IPython.display import Markdown, display
from nltk.corpus import stopwords
from nltk.tokenize import PunktTokenizer, RegexpTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Depuis la racine ou depuis le répertoire notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.embeddings import encode, load_wine_embeddings
from src.reference import load_wines

wines = load_wines()
descriptions = wines["description"].tolist()
print(f"{len(wines)} vins de référence, {wines['variety'].nunique()} cépages.")

600 vins de référence, 40 cépages.


## Requêtes et prétraitement

NLTK découpe chaque requête en phrases avec le tokenizer Punkt de sa langue. Pour TF-IDF et Word2Vec, on passe les phrases en minuscules, on sépare les mots à la ponctuation et on retire les mots-outils de la langue. Les accents sont conservés. SBERT encode la requête complète avec sa casse et sa ponctuation via son tokenizer.

In [2]:
# Les deux langues sont représentées. Les requêtes restent dans leur langue d'origine.
queries = [
    ("french", "Un rouge léger et fruité pour un apéro."),
    ("french", "Je cherche un blanc vif et minéral. Il accompagnera des huîtres."),
    ("english", "A full-bodied red with firm tannins, blackcurrant and oak for grilled steak."),
    ("french", "Un vin pétillant et sec. J’aime les notes de pomme verte et de citron."),
    ("english", "A sweet dessert wine with honey and apricot aromas. Ideally with a fresh finish."),
]

# Chaque ressource est téléchargée seulement si nécessaire.
for resource, name in [("tokenizers/punkt_tab", "punkt_tab"), ("corpora/stopwords", "stopwords")]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(name, quiet=True, raise_on_error=True)

languages = ("french", "english")
sentence_tokenizers = {language: PunktTokenizer(language) for language in languages}
stop_words = {language: set(stopwords.words(language)) for language in languages}
word_tokenizer = RegexpTokenizer(r"[^\W_]+")


def tokenize_words(text, language="english"):
    """Applique le même prétraitement lexical aux critiques et aux requêtes."""
    words = word_tokenizer.tokenize(text.lower())
    return [word for word in words if word not in stop_words[language]]


query_sentences = [sentence_tokenizers[language].tokenize(text) for language, text in queries]
query_tokens = [
    [word for sentence in sentences for word in tokenize_words(sentence, language)]
    for (language, _), sentences in zip(queries, query_sentences)
]
wine_tokens = [tokenize_words(text) for text in descriptions]
display(pd.DataFrame({
    "Requête": [text for _, text in queries],
    "Langue": ["FR" if language == "french" else "EN" for language, _ in queries],
    "Nombre de phrases": [len(sentences) for sentences in query_sentences],
}))

,Requête,Langue,Nombre de phrases
0,Un rouge léger et fruité pour un apéro.,FR,1
1,Je cherche un blanc vif et minéral. Il accompa...,FR,2
2,"A full-bodied red with firm tannins, blackcurr...",EN,1
3,Un vin pétillant et sec. J’aime les notes de p...,FR,2
4,A sweet dessert wine with honey and apricot ar...,EN,2


## TF-IDF

La base lexicale pondère les mots et les paires de mots adjacents selon leur fréquence dans les 600 critiques. Une requête sans terme connu reçoit un score nul pour chaque critique.

In [3]:
# Ajustement sur les critiques, puis projection des requêtes dans le même espace.
tfidf = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1, 2))
tfidf_wines = tfidf.fit_transform([" ".join(words) for words in wine_tokens])
tfidf_queries = tfidf.transform([" ".join(words) for words in query_tokens])
tfidf_scores = cosine_similarity(tfidf_queries, tfidf_wines)

## Word2Vec

On entraîne des vecteurs de mots de dimension 100 avec CBOW sur le corpus complet de critiques Wine Reviews fourni dans le dépôt. Le vecteur d'un texte est la moyenne des vecteurs de ses mots connus. En l'absence de mots connus, le vecteur est nul, et sa similarité cosinus est fixée à zéro. Une graine fixe et un seul worker stabilisent l'entraînement.

In [4]:
# Les descriptions du corpus complet servent à apprendre le vocabulaire de dégustation.
raw_reviews = pd.read_csv(ROOT / "data/raw/winemag_reviews.csv.gz", usecols=["description"])
training_tokens = [tokenize_words(text) for text in raw_reviews["description"].dropna()]
word2vec = Word2Vec(
    sentences=training_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    workers=1,
    seed=42,
    epochs=5,
    sg=0,
)
word_vectors = word2vec.wv
print(f"{len(training_tokens):,} critiques d'entraînement et {len(word_vectors):,} mots connus.")


def mean_word_vector(words):
    """Calcule la moyenne des vecteurs du vocabulaire Word2Vec présents dans le texte."""
    vectors = [word_vectors[word] for word in words if word in word_vectors]
    if not vectors:
        return np.zeros(word_vectors.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)


word2vec_wines = np.stack([mean_word_vector(words) for words in wine_tokens])
word2vec_queries = np.stack([mean_word_vector(words) for words in query_tokens])
word2vec_scores = cosine_similarity(word2vec_queries, word2vec_wines)

169,430 critiques d'entraînement et 14,492 mots connus.


## SBERT multilingue

[`paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) projette les textes français et anglais dans le même espace de 384 dimensions. Les vecteurs renvoyés par `encode()` sont de norme unitaire. Leur produit scalaire donne donc le cosinus. `load_wine_embeddings()` conserve les vecteurs dans `data/cache/`, selon l'ordre de `load_wines()`.

In [5]:
# Réutilise la matrice des vins et encode seulement les nouvelles requêtes.
sbert_wines = load_wine_embeddings()
sbert_queries = encode([text for _, text in queries])
sbert_scores = sbert_queries @ sbert_wines.T

assert sbert_wines.shape == (len(wines), 384)
assert np.allclose(np.linalg.norm(sbert_wines, axis=1), 1)
print(f"Matrice SBERT des vins : {sbert_wines.shape}.")

Matrice SBERT des vins : (600, 384).


## Comparaison des cinq premiers résultats

Chaque colonne donne le vin, son cépage, sa couleur et son score. Les scores servent à classer les vins au sein d'une méthode. Quand tous les scores d'une méthode sont nuls, ses cinq premières lignes sont ex æquo et l'ordre du catalogue départage les vins. Les mots connus affichés sous la requête aident à interpréter ces cas.

In [6]:
scores_by_method = {"TF-IDF": tfidf_scores, "Word2Vec": word2vec_scores, "SBERT": sbert_scores}
results = []
for query_index, (_, text) in enumerate(queries):
    # Le tri stable conserve l'ordre de load_wines() en cas d'égalité.
    for method, scores in scores_by_method.items():
        query_scores = scores[query_index]
        top_indices = np.argsort(-query_scores, kind="stable")[:5]
        for rank, wine_index in enumerate(top_indices, start=1):
            wine = wines.iloc[wine_index]
            results.append({
                "Requête": text,
                "Méthode": method,
                "Rang": rank,
                "Identifiant": wine["wine_id"],
                "Vin": wine["title"],
                "Cépage": wine["variety"],
                "Couleur": wine["color"],
                "Score": float(query_scores[wine_index]),
            })

# Ce tableau contient les 75 résultats détaillés et peut être réutilisé pour les slides.
results = pd.DataFrame(results)
assert len(results) == len(queries) * len(scores_by_method) * 5

for query_index, (_, text) in enumerate(queries):
    terms = query_tokens[query_index]
    tfidf_known = ", ".join(sorted({word for word in terms if word in tfidf.vocabulary_}))
    word2vec_known = ", ".join(sorted({word for word in terms if word in word_vectors}))
    display(Markdown(
        f"### {text}\n"
        f"Mots connus de TF-IDF **{tfidf_known or 'aucun'}** et de Word2Vec "
        f"**{word2vec_known or 'aucun'}**."
    ))
    rows = results.loc[results["Requête"] == text].copy()
    rows["Résultat"] = [
        f"{title} · {variety} ({color}) · {score:.3f}"
        for title, variety, color, score in zip(rows["Vin"], rows["Cépage"], rows["Couleur"], rows["Score"])
    ]
    table = rows.pivot(index="Rang", columns="Méthode", values="Résultat")
    with pd.option_context("display.max_colwidth", None):
        display(table[list(scores_by_method)])

### Un rouge léger et fruité pour un apéro.
Mots connus de TF-IDF **aucun** et de Word2Vec **rouge**.

Méthode,TF-IDF,Word2Vec,SBERT
Rang,,,
1,Dieter Meier 2010 Puro Corte d'Oro Red (Mendoza) · Bordeaux-style Red Blend (red) · 0.000,Conspire 2012 Sauvignon Blanc (Rutherford) · Sauvignon Blanc (white) · -0.054,Vite Colte 2016 Rossofuoco (Barbera d'Asti) · Barbera (red) · 0.774
2,Aymara 2013 Reserva Cabernet Franc (Mendoza) · Cabernet Franc (red) · 0.000,Bortolin NV Cartizze Dry (Valdobbiadene Superiore di Cartizze) · Glera (sparkling) · -0.159,Suberli 2014 Morellino di Scansano · Red Blend (red) · 0.746
3,Domaine Bousquet 2016 Made With Organically Grown Grapes Cabernet Sauvignon (Tupungato) · Cabernet Sauvignon (red) · 0.000,Clos du Val 2005 Cabernet Sauvignon (Stags Leap District) · Cabernet Sauvignon (red) · -0.194,El Cortijillo 2014 Tempranillo (La Mancha) · Tempranillo (red) · 0.703
4,Alta Vista 2006 Single Vineyard Temis Malbec (Uco Valley) · Malbec (red) · 0.000,Reynvaan Family Vineyards 2007 Queen's Road White White (Walla Walla Valley (WA)) · White Blend (white) · -0.223,Veglio 2010 Re Della Notte Extra Dry Sparkling (Piedmont) · Sparkling Blend (sparkling) · 0.701
5,Andeluna 2010 Malbec (Mendoza) · Malbec (red) · 0.000,Le Brun de Neuville NV Grande Sélection Brut (Champagne) · Champagne Blend (sparkling) · -0.249,Cascina Luisin 2012 Paolin (Barbaresco) · Nebbiolo (red) · 0.696


### Je cherche un blanc vif et minéral. Il accompagnera des huîtres.
Mots connus de TF-IDF **blanc** et de Word2Vec **blanc**.

Méthode,TF-IDF,Word2Vec,SBERT
Rang,,,
1,Château Calvimont 2015 Graves · Bordeaux-style White Blend (white) · 0.137,Le Brun de Neuville NV Grande Sélection Brut (Champagne) · Champagne Blend (sparkling) · 0.394,Telmo Rodríguez 2004 Lanzaga (Rioja) · Tempranillo Blend (red) · 0.454
2,Cheval Quancard 2015 Cuvée Clémence (Entre-Deux-Mers) · Bordeaux-style White Blend (white) · 0.131,McIntyre Vineyards NV L'Homme Qui Ris Sparkling (Santa Lucia Highlands) · Sparkling Blend (sparkling) · 0.321,Brokenwood 2010 Shiraz (Hunter Valley) · Shiraz (red) · 0.432
3,Sattlerhof 2006 Steirische Klassik Sauvignon Blanc (Südsteiermark) · Sauvignon Blanc (white) · 0.130,Leo Steen Wines 2011 Saini Farms Chenin Blanc (Dry Creek Valley) · Chenin Blanc (white) · 0.318,Louis M. Martini 2011 Cabernet Sauvignon (Alexander Valley) · Cabernet Sauvignon (red) · 0.431
4,Navardia 2014 Organic White (Rioja) · White Blend (white) · 0.117,Fiuza 2015 3 Castas White (Tejo) · Portuguese White (white) · 0.313,Burrell School Vineyards 2010 Valedictorian Red (Santa Cruz Mountains) · Bordeaux-style Red Blend (red) · 0.425
5,Terlan 2015 Terlaner Classico White (Alto Adige) · White Blend (white) · 0.117,Reynvaan Family Vineyards 2007 Queen's Road White White (Walla Walla Valley (WA)) · White Blend (white) · 0.303,Cálem NV 10-Years Old White (Port) · Port (dessert) · 0.419


### A full-bodied red with firm tannins, blackcurrant and oak for grilled steak.
Mots connus de TF-IDF **bodied, firm, full, grilled, oak, red, steak, tannins** et de Word2Vec **blackcurrant, bodied, firm, full, grilled, oak, red, steak, tannins**.

Méthode,TF-IDF,Word2Vec,SBERT
Rang,,,
1,Peju 2009 Reserve Cabernet Franc (Rutherford) · Cabernet Franc (red) · 0.222,Cuda Ridge Wines 2014 Cabernet Franc (Livermore Valley) · Cabernet Franc (red) · 0.869,Valley of the Moon 2010 Barbera (Sonoma County) · Barbera (red) · 0.579
2,Fanti 2012 Vallocchio (Brunello di Montalcino) · Sangiovese (red) · 0.144,Angove 2015 Family Crest Shiraz (McLaren Vale) · Shiraz (red) · 0.840,Chatzivariti 2010 Eurynome Red Dry Wine Xinomavro-Negoska Red (Goumenissa) · Red Blend (red) · 0.562
3,Fratelli Alessandria 2012 Barolo · Nebbiolo (red) · 0.119,Muga 2005 Torre Muga (Rioja) · Tempranillo Blend (red) · 0.837,Ridge 2008 York Creek Zinfandel (California) · Zinfandel (red) · 0.553
4,Viña Bisquertt 2013 La Joya Gran Reserva Carmenère (Colchagua Valley) · Carmenère (red) · 0.113,Fanti 2012 Vallocchio (Brunello di Montalcino) · Sangiovese (red) · 0.834,Peju 2009 Reserve Cabernet Franc (Rutherford) · Cabernet Franc (red) · 0.551
5,Cuda Ridge Wines 2014 Cabernet Franc (Livermore Valley) · Cabernet Franc (red) · 0.112,Jaxon Keys 2013 Estate Allie Key's Vineyard Petite Sirah (Mendocino) · Petite Sirah (red) · 0.823,Laberinto 2013 Malbec (Mendoza) · Malbec (red) · 0.549


### Un vin pétillant et sec. J’aime les notes de pomme verte et de citron.
Mots connus de TF-IDF **notes, vin** et de Word2Vec **citron, notes, pétillant, sec, vin**.

Méthode,TF-IDF,Word2Vec,SBERT
Rang,,,
1,Querciavalle 1998 Vin Santo del Chianti Classico · White Blend (white) · 0.148,Cantina Progettidivini 2010 Millesimato (Conegliano Valdobbiadene Prosecco Superiore) · Glera (sparkling) · 0.518,Vranken 2000 Demoiselle Premier Cru Millésimé Brut (Champagne) · Champagne Blend (sparkling) · 0.799
2,"Tommasi 2004 Cà Florian (Amarone della Valpolicella Classico) · Corvina, Rondinella, Molinara (red) · 0.056",Sangervasio 2005 Recinaio White (Colli Etruria Centrale) · White Blend (white) · 0.518,Rabl 2011 Spiegl Grüner Veltliner (Kamptal) · Grüner Veltliner (white) · 0.756
3,Aymara 2013 Reserva Cabernet Franc (Mendoza) · Cabernet Franc (red) · 0.051,"Ca' Rugate 2011 Rio Albo (Valpolicella) · Corvina, Rondinella, Molinara (red) · 0.505",Dorigo 2014 Pinot Grigio (Friuli Colli Orientali) · Pinot Grigio (white) · 0.756
4,Vieil Armand 2015 Tradition Pinot Gris (Alsace) · Pinot Gris (white) · 0.038,Parés Baltà NV Brut Sparkling (Cava) · Sparkling Blend (sparkling) · 0.457,Aveleda 2012 Grinalda White (Vinho Verde) · Portuguese White (white) · 0.751
5,Raphael Dal Bo 2014 Brut Millesimato (Conegliano Valdobbiadene Prosecco Superiore) · Glera (sparkling) · 0.036,Aymara 2013 Reserva Cabernet Franc (Mendoza) · Cabernet Franc (red) · 0.453,Herdade da Comporta 2014 White (Península de Setúbal) · Portuguese White (white) · 0.746


### A sweet dessert wine with honey and apricot aromas. Ideally with a fresh finish.
Mots connus de TF-IDF **apricot, aromas, dessert, finish, fresh, honey, sweet, wine** et de Word2Vec **apricot, aromas, dessert, finish, fresh, honey, ideally, sweet, wine**.

Méthode,TF-IDF,Word2Vec,SBERT
Rang,,,
1,Balletto 2009 Vin de Paille Estate Pinot Gris (Russian River Valley) · Pinot Gris (white) · 0.393,Balletto 2009 Vin de Paille Estate Pinot Gris (Russian River Valley) · Pinot Gris (white) · 0.887,Joseph Filippi NV Dulce Lagrimas de Elisa Chenin Blanc (California) · Chenin Blanc (white) · 0.847
2,Rivka Simone 2007 Viognier (Russian River Valley) · Viognier (white) · 0.222,Joseph Filippi NV Dulce Lagrimas de Elisa Chenin Blanc (California) · Chenin Blanc (white) · 0.869,Balletto 2009 Vin de Paille Estate Pinot Gris (Russian River Valley) · Pinot Gris (white) · 0.811
3,Clos Solène 2010 Sweet Clémentine Grenache (Paso Robles) · Grenache (red) · 0.194,Pride Mountain 2007 Viognier (Sonoma County) · Viognier (white) · 0.849,De Tierra 2008 Five by Five Red (Monterey) · Bordeaux-style Red Blend (red) · 0.809
4,Querciavalle 1998 Vin Santo del Chianti Classico · White Blend (white) · 0.181,Rivka Simone 2007 Viognier (Russian River Valley) · Viognier (white) · 0.842,Château Les Hauts de Palette 2015 Château du Barail Rosé (Bordeaux Rosé) · Rosé (rose) · 0.799
5,Sangervasio 2005 Recinaio White (Colli Etruria Centrale) · White Blend (white) · 0.148,Querciavalle 1998 Vin Santo del Chianti Classico · White Blend (white) · 0.830,Cedarville Vineyard 2015 Estate Viognier (El Dorado) · Viognier (white) · 0.794


## Interprétation

- Pour le rouge léger et fruité, TF-IDF donne 600 scores nuls. Word2Vec reconnaît seulement « rouge » et classe un Sauvignon Blanc en tête. SBERT propose un Barbera rouge en premier.
- Pour le vin pétillant et sec, Word2Vec et SBERT placent un effervescent en tête. Les trois méthodes placent un rouge en premier pour la requête anglaise sur le vin corsé et tannique.
- Pour le blanc vif et minéral destiné aux huîtres, SBERT classe un rouge de Rioja en tête. Le lot 3 pourra appliquer un filtre de couleur et utiliser la table d'accords pour ce type de requête.

Ces cinq exemples donnent une comparaison qualitative. Une évaluation sur davantage de requêtes est prévue dans le lot 6.